# GridSpark AI – Notebook 01: Feasibility Matrix

**Purpose:** Verify all five open-data layers needed for the Lane County, Oregon wildfire ignition risk prototype before any model training begins.

**Fire event:** Holiday Farm Fire · Lane County, OR · 2020-09-07

> **Decision-support framing:** This prototype is a decision-support tool for inspection prioritization, vegetation management, and risk screening. It does NOT automate infrastructure shutoffs or replace utility operational judgment.

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import json
import pandas as pd
import geopandas as gpd
import folium
from pathlib import Path
from IPython.display import display, Markdown

from config import (
    IGNITION_LAT, IGNITION_LON, IGNITION_DATE, FIRE_NAME, COUNTY,
    BUFFER_DISTANCES_M, AOI_GEOJSON, FEASIBILITY_CSV, FEASIBILITY_MD,
)

print(f'Fire: {FIRE_NAME}')
print(f'County: {COUNTY}')
print(f'Ignition anchor: ({IGNITION_LAT}, {IGNITION_LON})')
print(f'Ignition date: {IGNITION_DATE}')

## Step 1 – Initialize the feasibility matrix

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '../scripts/build_feasibility_matrix.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 2 – Generate AOI GeoJSON

In [ ]:
result = subprocess.run([sys.executable, '../scripts/generate_aoi.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 3 – Visualize AOI on interactive map

In [ ]:
m = folium.Map(location=[IGNITION_LAT, IGNITION_LON], zoom_start=11, tiles='CartoDB positron')

colors = {1000: 'red', 3000: 'orange', 5000: 'blue'}

if Path(AOI_GEOJSON).exists():
    with open(AOI_GEOJSON) as f:
        aoi = json.load(f)
    for feat in aoi['features']:
        buf = feat['properties'].get('buffer_m', 0)
        if buf == 0:
            folium.Marker(
                [IGNITION_LAT, IGNITION_LON],
                popup='Holiday Farm Fire ignition anchor (approximate)',
                icon=folium.Icon(color='red', icon='fire', prefix='fa')
            ).add_to(m)
        else:
            folium.GeoJson(
                feat,
                style_function=lambda f, c=colors.get(buf, 'gray'): {
                    'fillColor': c, 'color': c, 'weight': 2, 'fillOpacity': 0.1
                },
                tooltip=f'{buf/1000:.0f} km AOI'
            ).add_to(m)
else:
    print('AOI GeoJSON not found — run generate_aoi.py first.')
    folium.Marker([IGNITION_LAT, IGNITION_LON], popup='Ignition anchor').add_to(m)

m

## Step 4 – Run all verification scripts

In [ ]:
scripts = [
    '../scripts/verify_hifld.py',
    '../scripts/verify_fire_labels.py',
    '../scripts/verify_fire_perimeter.py',
    '../scripts/verify_noaa_isd.py',
]

for script in scripts:
    print(f'\n{'='*60}')
    print(f'Running: {script}')
    print('='*60)
    res = subprocess.run([sys.executable, script], capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print('LOG:', res.stderr[-2000:])  # last 2000 chars of log

## Step 5 – Display current feasibility matrix

In [ ]:
if Path(FEASIBILITY_CSV).exists():
    df = pd.read_csv(FEASIBILITY_CSV)
    display(df.T.rename(columns={0: 'Value'}))
else:
    print('Feasibility matrix CSV not found.')

In [ ]:
if Path(FEASIBILITY_MD).exists():
    with open(FEASIBILITY_MD) as f:
        display(Markdown(f.read()))
else:
    print('Feasibility matrix Markdown not found.')

## Step 6 – Build training table schema

In [ ]:
res = subprocess.run([sys.executable, '../scripts/build_training_table_schema.py'], capture_output=True, text=True)
print(res.stdout)

from config import MODEL_SCHEMA_CSV
if Path(MODEL_SCHEMA_CSV).exists():
    display(pd.read_csv(MODEL_SCHEMA_CSV))

---

## Feasibility gate decision

**Do not proceed to model training until all five layers show `Pass`.**

| Layer | Required for training? | Can approximate? |
|---|---|---|
| HIFLD transmission lines | Yes | Partial (buffer corridors) |
| FPA-FOD ignition points | Yes (labels) | Yes (with buffer sensitivity) |
| MTBS/WFIGS fire perimeter | Strongly recommended | Use FPA-FOD as fallback |
| Sentinel-2 pre-fire imagery | Yes (features) | No |
| NOAA ISD weather | Yes (features) | Partial (ERA5 reanalysis) |

If any layer returns `Manual Check`, complete the manual download before continuing to Notebook 02.